# Memory Mosaics in JAX: Fig. 7 subplot

Trains a 1-block GPT-2 and a 1-block Memory Mosaic with identical data and optimizer settings
and plots their loss curves, reproducing (in miniature) the paper's Fig. 7 claim that a Memory Mosaic
trains as well as GPT-2 with a single block. See the repo README for the architecture and scope.

Runtime → Change runtime type → **GPU**, then run top to bottom.

In [ ]:
# Colab already ships JAX with CUDA; reinstalling jax[cuda12] registers the GPU plugin twice.
!pip install -q optax tokenizers datasets matplotlib
import os, sys
if not os.path.exists("memory-mosaics-jax"):
    !git clone -q https://github.com/rohitium/memory-mosaics-jax.git
sys.path.insert(0, "memory-mosaics-jax/src")

import jax
print(jax.devices())

## Configs

`pmem_size ≈ 3.5 × n_embd` gives the Mosaic block about the same parameter count as a GPT-2 block.

In [ ]:
from mm_jax import config, gpt2, memory_mosaic, train
from mm_jax.nn import count_params

g_cfg = config.ModelConfig(n_layer=1, n_head=4, n_embd=128, block_size=128)
m_cfg = config.MosaicConfig(n_layer=1, n_head=4, n_embd=128, block_size=128, pmem_size=448)
t_cfg = config.TrainConfig(batch_size=16, block_size=128, max_steps=400)

key = jax.random.PRNGKey(0)
g_params, m_params = gpt2.init(key, g_cfg), memory_mosaic.init(key, m_cfg)
for name, p in [("GPT-2", g_params), ("Mosaic", m_params)]:
    print(f"{name:6s} block params: {count_params(p['blocks']):,}   total: {count_params(p):,}")

## Data

~1M characters of TinyStories, GPT-2 BPE tokenized, split 95/5 into train/val.

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer

parts, n_chars = [], 0
for ex in load_dataset("roneneldan/TinyStories", split="train", streaming=True):
    parts.append(ex["text"])
    n_chars += len(ex["text"])
    if n_chars >= 1_000_000:
        break
ids = Tokenizer.from_pretrained("gpt2").encode("\n".join(parts)).ids
split = int(0.95 * len(ids))
train_b, val_b = train.TextBatcher(ids[:split], seed=0), train.TextBatcher(ids[split:], seed=1)
print(f"{split:,} train / {len(ids) - split:,} val tokens")

## Train both models

In [ ]:
import time

def run(name, module, cfg, params):
    train_step, eval_step, state = train.make_trainer(module.apply, params, cfg, t_cfg)
    train_losses, val_steps, val_losses = [], [], []
    t0 = time.time()
    for i in range(1, t_cfg.max_steps + 1):
        state, loss = train_step(state, train_b.next_batch(t_cfg.batch_size, t_cfg.block_size))
        train_losses.append(float(loss))
        if i % t_cfg.eval_every == 0:
            val_steps.append(i)
            val_losses.append(train.eval_loss(eval_step, state["params"], val_b, t_cfg))
            print(f"[{name}] step {i:4d}  train {float(loss):.3f}  val {val_losses[-1]:.3f}  ({time.time() - t0:.0f}s)")
    return train_losses, val_steps, val_losses

g_run = run("gpt2  ", gpt2, g_cfg, g_params)
m_run = run("mosaic", memory_mosaic, m_cfg, m_params)

## Plot

If the two curves track each other, the subplot is reproduced.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

W = 15
steps = np.arange(1, t_cfg.max_steps + 1)
plt.figure(figsize=(8, 5))
for (train_l, val_s, val_l), label, c in [(g_run, "GPT-2", "C0"), (m_run, "Memory Mosaic", "C1")]:
    plt.plot(steps, train_l, alpha=0.2, color=c)
    plt.plot(steps[W - 1:], np.convolve(train_l, np.ones(W) / W, mode="valid"), color=c, lw=2,
             label=f"{label} (1 block), train")
    plt.scatter(val_s, val_l, color=c, s=40, zorder=5, edgecolor="white", label=f"{label}, val")
plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title("1-block Memory Mosaic vs 1-block GPT-2 (TinyStories)")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("fig7_subplot.png", dpi=150, bbox_inches="tight")
plt.show()